In [8]:
from recbole.quick_start import load_data_and_model, run_recbole
import torch
import pandas as pd


import torch
from recbole.config import Config
from recbole.data import create_dataset, data_preparation




from recbole.model.general_recommender import NeuMF
from recbole.trainer import Trainer
from recbole.utils import get_model, get_trainer, init_seed, init_logger
from collections import defaultdict
import os
from recbole.quick_start import load_data_and_model
import rbo as rbo_lib

import rbo


In [1]:
import torch
import numpy as np
import pandas as pd
from collections import defaultdict

# ── 0. LOAD MODEL AND DATASET ───────────────────────────────────────────────
from recbole.quick_start import load_data_and_model

config, sasrec_model, dataset, train_data, valid_data, test_data = load_data_and_model(
    model_file='./saved/SASRec-lastfm_final.pth'
)
#SASRec-beeradvocate_256.pth
sasrec_model.eval()
device = config['device']

/home/mvarasteh/post-hoc/recbole/quick_start/quick_start.py:249: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_file, map_location=torch.device(

In [5]:
import yaml
from recbole.quick_start import load_data_and_model

with open('YAML_files/CBM_config.yaml', 'r') as f:
    config_overrides = yaml.safe_load(f)

config_overrides['train_neg_sample_args']={'distribution': 'none', 'sample_num': 'none', 'alpha': 'none', 'dynamic': False, 'candidate_num': 0}

config, cbm_sasrec, dataset, train_data, valid_data, test_data = load_data_and_model(
    config_dict=config_overrides, 
        model_file="./saved/SASRec_CBM-Jul-20-2026_18-18-44.pth")


## SASRec_CBM-Jul-20-2026_17-52-19.pth. (adding loss_suff term to the loss fucntion)
## SASRec_CBM-Jul-20-2026_12-58-13.pth (v2.1)
## SASRec_CBM-Jul-19-2026_21-22-31.pth (pop>80, niche <20 same structure as befor)
## SASRec_CBM-Jul-20-2026_10-38-21.pth (reg is 1 and for sigmoid function, free channel is 64)
## SASRec_CBM-Jul-19-2026_18-41-45.pth (for sigmoid function and free channel 64 is a good result regularization term (0.1) for n_free)
##SASRec_CBM-Jul-19-2026_18-04-46.pth" (for sigmoid function and free channel 64 is a good result)
## SASRec_CBM-Jul-16-2026_17-56-02.pth. (this one is with 64 free size and adding all terms in losss functiona nd with softmax actiation fucntion)
# SASRec_CBM-Jul-11-2026_15-59-58.pth
cbm_sasrec.eval()
device = config['device']

/home/mvarasteh/post-hoc/recbole/quick_start/quick_start.py:249: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_file, map_location=torch.device(

[build] Dataset: lastfm
[build] Building concept lookups...
  pop_set:   1,370 items (≥ 220 interactions)
  mid_set:   4,059 items
  niche_set: 1,418 items (1–47 interactions)
  Items in metadata: 6,849, matched to internal IDs: 6,849
  N_CONCEPTS = 22  (16 genres + 6 scalars)
[build] Computing per-item concept vectors for 6,850 items...

[build] Saved concept matrix (6850, 22) → ./dataset/lastfm/saved_concept_individual_items.pkl

[build] Diagnostics — items with NO ...
  ... primary features : 1 (should be ~1, padding)
  ... tier features    : 3 (items w/ 0 interactions)
  ... secondary feats  : 1 (items missing metadata)
  mean genres/item     : 1.01

[build] Spot check — item 1:
  Rock                      1.0000
  mid                       1.0000
  modern                    1.0000
[CBM] 6850 items × 22 concepts  +  64 free dims


In [6]:
W = cbm_sasrec.reconstructor[0].weight.detach()  # [128, 104]
b = cbm_sasrec.reconstructor[0].bias.detach()     # [128]

col_norms = W.norm(dim=0)   # [88] — one per input dim (concepts + z)
row_norms = W.norm(dim=1)   # [128] — one per output dim

# split by your concept layout (24 concepts, 64 free latents)
n_concepts = 24
concept_norms = col_norms[:n_concepts]
z_norms      = col_norms[n_concepts:]

print(f"concept col-norm mean: {concept_norms.mean():.4f}  max: {concept_norms.max():.4f}")
print(f"z       col-norm mean: {z_norms.mean():.4f}  max: {z_norms.max():.4f}")

concept col-norm mean: 28.4671  max: 55.4079
z       col-norm mean: 7.3582  max: 7.7548


In [ ]:
import numpy as np

W = cbm_sasrec.reconstructor[0].weight.detach().cpu().numpy()   # [128, 24+64]
# if reconstructor is a plain nn.Linear (v2), use: model.reconstructor.weight

n_c = cbm_sasrec.n_concepts          # 24
W_c, W_z = W[:, :n_c], W[:, n_c:]

# use the cohort activations you already captured, e.g. the pop cohort:
c = c_by_cohort["pop"]          # numpy [N, 24]
z = z_by_cohort["pop"]          # numpy [N, 64]

infl_c = (np.linalg.norm(W_c, axis=0) * c.std(0)).sum()
infl_z = (np.linalg.norm(W_z, axis=0) * z.std(0)).sum()
print(f"concept influence {infl_c:.2f}  vs  z influence {infl_z:.2f}  "
      f"(z share: {infl_z / (infl_c + infl_z):.1%})")

## RBO comparison

In [6]:


@torch.no_grad()
def get_top_k_recommendations(model, data_loader, k=20, exclude_seen=True):
    """
    Returns {internal_user_id: LongTensor[k]} of top-k recommended INTERNAL
    item ids, ranked best-first.

    Compatible with compare_recommendations(): each value is a tensor on which
    .numpy().tolist() works.

    exclude_seen=True masks items already in the user's history before ranking,
    matching RecBole's default full-sort eval protocol. Set False only if your
    reported Recall/NDCG were computed WITHOUT seen-item masking.
    """
    model.eval()
    device = next(model.parameters()).device
    recs = {}

    for batch in data_loader:
        interaction = batch[0] if isinstance(batch, (list, tuple)) else batch
        interaction = interaction.to(device)

        # [B, n_items] full-catalog scores
        scores = model.full_sort_predict(interaction)
        B = interaction[model.ITEM_SEQ].shape[0]
        scores = scores.view(B, -1).clone()

        # mask padding item (internal id 0)
        scores[:, 0] = -float('inf')

        if exclude_seen:
            item_seq = interaction[model.ITEM_SEQ]
            for i in range(B):
                seen = item_seq[i][item_seq[i] != 0]
                scores[i, seen] = -float('inf')

        topk = torch.topk(scores, k, dim=1).indices          # [B, k] internal ids

        uids = interaction[model.USER_ID].tolist()
        for u, row in zip(uids, topk):
            recs[u] = row.detach().cpu()

    return recs

In [9]:


def compare_recommendations(recs_sasrec, recs_cbm, k=20, p=0.9):
    """
    Compare top-k recommendations between two models.
    Computes per-user RBO, Jaccard, and overlap count.
    """
    rbo_scores     = []
    jaccard_scores = []
    overlap_counts = []
    
    common_users = set(recs_sasrec.keys()) & set(recs_cbm.keys())
    
    for user_idx in sorted(common_users):
        list_a = recs_sasrec[user_idx].numpy().tolist()
        list_b = recs_cbm[user_idx].numpy().tolist()
        
        set_a = set(list_a)
        set_b = set(list_b)
        
        intersection = len(set_a & set_b)
        union        = len(set_a | set_b)
        
        jaccard = intersection / union if union > 0 else 0
        rbo_val     = rbo.RankingSimilarity(list_a, list_b).rbo()
        
        rbo_scores.append(rbo_val )
        jaccard_scores.append(jaccard)
        overlap_counts.append(intersection)
    
    rbo_scores     = np.array(rbo_scores)
    jaccard_scores = np.array(jaccard_scores)
    overlap_counts = np.array(overlap_counts)
    
    print(f"\n{'='*60}")
    print(f"Top-{k} Recommendation Comparison: SASRec vs CBM")
    print(f"{'='*60}")
    print(f"  Users compared:        {len(common_users)}")
    print(f"  Mean RBO (p={p}):       {rbo_scores.mean():.4f}")
    print(f"  Median RBO:            {np.median(rbo_scores):.4f}")
    print(f"  Mean Jaccard:          {jaccard_scores.mean():.4f}")
    print(f"  Median Jaccard:        {np.median(jaccard_scores):.4f}")
    print(f"  Mean overlap (/{k}):    {overlap_counts.mean():.1f}")
    print(f"  Median overlap (/{k}):  {np.median(overlap_counts):.1f}")
    print(f"  Perfect match (={k}):   {(overlap_counts == k).sum()} users")
    print(f"  Zero overlap (=0):     {(overlap_counts == 0).sum()} users")
    
    print(f"\n  Overlap distribution:")
    for threshold in [0, 5, 10, 15, 20]:
        count = (overlap_counts >= threshold).sum()
        pct = 100 * count / len(overlap_counts)
        print(f"    ≥{threshold:2d} items overlap:  {count:5d} users ({pct:.1f}%)")
    
    return rbo_scores, jaccard_scores, overlap_counts


recs_sasrec = get_top_k_recommendations(sasrec_model, test_data, k=20)
recs_cbm    = get_top_k_recommendations(cbm_sasrec, test_data, k=20)

rbo_scores, jaccard_scores, overlap_counts = compare_recommendations(recs_sasrec, recs_cbm, k=20, p=0.9)


Top-20 Recommendation Comparison: SASRec vs CBM
  Users compared:        26904
  Mean RBO (p=0.9):       0.8351
  Median RBO:            0.8442
  Mean Jaccard:          0.7675
  Median Jaccard:        0.7391
  Mean overlap (/20):    17.3
  Median overlap (/20):  17.0
  Perfect match (=20):   627 users
  Zero overlap (=0):     0 users

  Overlap distribution:
    ≥ 0 items overlap:  26904 users (100.0%)
    ≥ 5 items overlap:  26896 users (100.0%)
    ≥10 items overlap:  26863 users (99.8%)
    ≥15 items overlap:  25989 users (96.6%)
    ≥20 items overlap:    627 users (2.3%)


## H and H_hat similarity

In [10]:
import numpy as np
import torch
import torch.nn.functional as F
from scipy.stats import spearmanr


@torch.no_grad()
def collect_h_hhat(cbm_model, data_loader, device=None):
    """
    Run the CBM over a data loader and collect the frozen-SASRec embedding `h`
    and the reconstructed embedding `h_hat` for every batch.

    Returns two tensors: H [N, hidden], H_hat [N, hidden]  (on CPU).
    """
    if device is None:
        device = next(cbm_model.parameters()).device

    cbm_model.eval()
    H, H_hat = [], []

    for batch in data_loader:
        # RecBole loaders yield an Interaction (or a tuple where it's first)

        interaction = batch[0] if isinstance(batch, (list, tuple)) else batch
        interaction = interaction.to(device)

        item_seq     = interaction[cbm_model.ITEM_SEQ]
        item_seq_len = interaction[cbm_model.ITEM_SEQ_LEN]
        

        h, c_hat,z, h_hat = cbm_model.forward(item_seq, item_seq_len)

        #h, c_hat, h_hat = cbm_model.forward(item_seq, item_seq_len)
        H.append(h.detach().cpu())
        H_hat.append(h_hat.detach().cpu())

    return torch.cat(H, dim=0), torch.cat(H_hat, dim=0)


def embedding_similarity_report(H, H_hat, item_embeddings=None, n_score_sample=2000):
    """
    H, H_hat : [N, hidden] tensors (the same N rows, aligned).
    item_embeddings : optional [n_items, hidden] tensor — if given, also reports
                      how similarly h and h_hat *rank* items (Spearman of scores).

    Prints a report and returns a dict of metrics.
    """
    assert H.shape == H_hat.shape, f"shape mismatch {H.shape} vs {H_hat.shape}"
    N, D = H.shape
    Hf, Hhf = H.float(), H_hat.float()

    # ── 1. Per-row cosine similarity (direction) ─────────────────────────────
    cos = F.cosine_similarity(Hf, Hhf, dim=-1)              # [N]

  
    

    # ── 5. Coefficient of determination R^2 (variance explained) ─────────────
    ss_res = ((Hf - Hhf) ** 2).sum()
    ss_tot = ((Hf - Hf.mean()) ** 2).sum()

    metrics = {
        'cosine_mean':      cos.mean().item(),
        'cosine_median':    cos.median().item(),
        'cosine_std':       cos.std().item(),
      
    }

    print(f"\n{'='*60}")
    print(f"Embedding similarity:  h  vs  h_hat   (N={N}, dim={D})")
    print(f"{'='*60}")
    print(f"  Cosine    mean={metrics['cosine_mean']:.4f}  "
          f"median={metrics['cosine_median']:.4f}  "
          f"std={metrics['cosine_std']:.4f}")
   



    print(f"{'='*60}\n")
    return metrics

In [11]:

H, H_hat = collect_h_hhat(cbm_sasrec, test_data)

metrics = embedding_similarity_report(
    H, H_hat,
    item_embeddings=cbm_sasrec.item_embedding.weight,   # ties it back to ranking
)

ValueError: too many values to unpack (expected 4)

## using two concepts for steering

In [11]:
def evaluate_with_recbole_sasrec(model, config, train_data, test_data):
    """Set steering on the model and run RecBole eval. Tier/genre/year exposure
    metrics are reported automatically alongside standard rec metrics."""
   

    

    trainer = Trainer(config, model)
    trainer.eval_collector = Collector(config)
    trainer.evaluator      = Evaluator(config)
    trainer.eval_collector.data_collect(train_data)

    result = trainer.evaluate(test_data, load_best_model=False, show_progress=False)

 
    return result
def evaluate_with_recbole(cbm, config, train_data, test_data,
                          concept_idx=None, scale=1.0):
    """Set steering on the model and run RecBole eval. Tier/genre/year exposure
    metrics are reported automatically alongside standard rec metrics."""
    cbm.steer_concept_idx = concept_idx
    #config['steer_concept_idx'] = concept_idx
    cbm.steer_scale       = scale
    #config['steer_scale']       = scale
   

    trainer = Trainer(config, cbm)
    trainer.eval_collector = Collector(config)
    trainer.evaluator      = Evaluator(config)
    trainer.eval_collector.data_collect(train_data)

    result = trainer.evaluate(test_data, load_best_model=False, show_progress=False)

    cbm.steer_concept_idx = None
    #config['steer_concept_idx'] = None
    cbm.steer_scale       = 1.0
    #config['steer_scale' ]      = 1.0
    return result

## Scaling Factor

In [28]:
import types
import torch
import torch.nn.functional as F

from recbole.utils.build_concepts import build_lookups
from recbole.evaluator import Collector, Evaluator

dataset_name = "lastfm"

# ═══════════════════════════════════════════════════════════════
# Patched forward: multi-concept intervention + activation capture
# ═══════════════════════════════════════════════════════════════
def patched_forward(self, item_seq, item_seq_len):
    h = self._encode(item_seq, item_seq_len)
    #c = self._lookup_concepts(item_seq)

    #c_hat = self.concept_predictor(h)
    logits = self.concept_predictor(h)
    #c_hat=torch.sigmoid(logits)
    #print("c_hat_bef_pop:", c_hat_tmp[:,18].mean())
    #print("c_hat_bef_mid:", c_hat_tmp[:,19].mean())
    z = self.latent_predictor(h)
    # apply interventions (eval only)
    if getattr(self, '_interventions', None) and not self.training:
        logits = logits.clone()
        #mask = logits[:, 16] > 0.8
        #mask = logits[:, 18] > logits[:, 19] # e.g. 0.0 → p(pop) > 0.5
        #logits[mask,19] += (3)

        for idx, scale in self._interventions.items():
            #print(f"idx {idx}")
            print("before",idx, logits[:,idx].mean())
            #if idx==18:

            logits[:,idx] += (scale)

            #elif idx==20:
                #logits[:,idx] += (scale)

            #print("after:", logits[:,idx].mean())
    c_hat=torch.sigmoid(logits)
    #print(f"std_of_c_hat {c_hat.std(dim=0)}")
    print("c_hat_after_pop:", c_hat[:,16].mean())
    print("c_hat_after_mid:", c_hat[:,17].mean())

    #c_hat = torch.cat([
        #F.softmax(logits[:, :18],    dim=-1),   # beer styles
        #F.softmax(logits[:, 18:21], dim=-1),   # popularity tiers
        #F.softmax(logits[:, 21:24], dim=-1),   # era bins
        #], dim=-1)



        
    

    

        #c_hat = torch.cat([
        #F.softmax(c_hat[:, :16],    dim=-1),   # beer styles
        #F.softmax(c_hat[:, 16:19], dim=-1),   # popularity tiers
       # F.softmax(c_hat[:, 19:22], dim=-1),   # era bins
   # ], dim=-1)
    
    #c_hat = torch.cat([
        #F.softmax(logits[:, :16],    dim=-1),   # beer styles
        #F.softmax(logits[:, 16:19], dim=-1),   # popularity tiers
       # F.softmax(logits[:, 19:22], dim=-1),   # era bins
    #], dim=-1)
    # capture activations (eval only, when enabled)
   

    bottleneck = torch.cat([c_hat, z], dim=-1)
    h_hat = self.reconstructor(bottleneck)
    return h, c_hat, z, h_hat






# ═══════════════════════════════════════════════════════════════
# Config
# ═══════════════════════════════════════════════════════════════
config['metrics']      = ['Recall', 'NDCG', 'MRR', 'Hit',
                          'ItemCoverage', 'GiniIndex',
                          'AveragePopularity',
                          'TierExposure', 'ItemCoverageN']
config['topk']         = [10,20]
config['valid_metric'] = 'NDCG@10'
config['eval_args']    = {'mode': 'full'}
config['tail_ratio']   = 0.2

L = build_lookups(dataset, config['dataset'])
config['popularity_pool'] = list(L['pop_set'])
config['mid_pool']        = list(L['mid_set'])
config['niche_pool']      = list(L['niche_set'])
config['item_to_genres']=dict(L['item_to_genres'])
config['genre_list']=list(L['genre_concepts'])

config['item_to_era']=dict(L['item_to_era'])


#c_hat_base, c_base = stop_capture(sasrec_model)


# ═══════════════════════════════════════════════════════════════
# Patch model (before any eval)
# ═══════════════════════════════════════════════════════════════
cbm_sasrec.steer_concept_idx = None
cbm_sasrec.steer_scale = 1.0
cbm_sasrec.forward = types.MethodType(patched_forward, cbm_sasrec)

concept_idx_pop = cbm_sasrec.concept_names.index('popularity')
concept_idx_mid = cbm_sasrec.concept_names.index('niche')
#print(f"concept_idx_pop{concept_idx_pop}")
#print(f"concept_idx_mid{concept_idx_mid}")

# ═══════════════════════════════════════════════════════════════
# Baseline (no steering)
# ═══════════════════════════════════════════════════════════════
cbm_sasrec._interventions = {}
#start_capture(cbm_sasrec)
result_baseline = evaluate_with_recbole(cbm_sasrec, config, train_data, test_data)

# ═══════════════════════════════════════════════════════════════
# Steered
# ═══════════════════════════════════════════════════════════════
cbm_sasrec._interventions = {
    concept_idx_pop: -0.9,
    concept_idx_mid: 2
}
#start_capture(cbm_sasrec)
result_steered = evaluate_with_recbole(cbm_sasrec, config, train_data, test_data)
#c_hat_steer, c_steer = stop_capture(cbm_sasrec)

cbm_sasrec._interventions = {}



# ═══════════════════════════════════════════════════════════════
# Results table
# ═══════════════════════════════════════════════════════════════
gap_before = result_baseline['pop_rate@10'] - result_baseline['mid_rate@10']
gap_after  = result_steered['pop_rate@10'] - result_steered['mid_rate@10']
print(f"\npop-mid gap: {gap_before:.4f} → {gap_after:.4f}")

print(f"\n{'Metric':<35} {'Baseline':>12} {'Steered':>12} {'Δ':>12}")
print("─" * 73)
for k in sorted(result_baseline.keys()):
    b, s = result_baseline[k], result_steered[k]
    print(f"{k:<35} {b:>12.4f} {s:>12.4f} {s - b:>+12.4f}")

  pop_set:   1,370 items (≥ 220 interactions)
  mid_set:   4,059 items
  niche_set: 1,418 items (1–47 interactions)
  Items in metadata: 6,849, matched to internal IDs: 6,849


  N_CONCEPTS = 22  (16 genres + 6 scalars)
c_hat_after_pop: tensor(0.4979, device='cuda:0')
c_hat_after_mid: tensor(0.3960, device='cuda:0')
c_hat_after_pop: tensor(0.4910, device='cuda:0')
c_hat_after_mid: tensor(0.4023, device='cuda:0')
c_hat_after_pop: tensor(0.4975, device='cuda:0')
c_hat_after_mid: tensor(0.3979, device='cuda:0')
c_hat_after_pop: tensor(0.5117, device='cuda:0')
c_hat_after_mid: tensor(0.3877, device='cuda:0')
c_hat_after_pop: tensor(0.5126, device='cuda:0')
c_hat_after_mid: tensor(0.3900, device='cuda:0')
c_hat_after_pop: tensor(0.5154, device='cuda:0')
c_hat_after_mid: tensor(0.3907, device='cuda:0')
c_hat_after_pop: tensor(0.4974, device='cuda:0')
c_hat_after_mid: tensor(0.4129, device='cuda:0')
before 16 tensor(0.0427, device='cuda:0')
before 18 tensor(-3.5319, device='cuda:0')
c_hat_after_pop: tensor(0.3188, device='cuda:0')
c_hat_after_mid: tensor(0.3960, device='cuda:0')
before 16 tensor(0.0057, device='cuda:0')
before 18 tensor(-3.5446, device='cuda:0')
c_h

In [24]:
import types
import itertools
import pandas as pd
import torch

from recbole.utils.build_concepts import build_lookups


dataset_name="lastfm"




# ── config setup (same as before) ──────────────────────────────
config['metrics']      = ['Recall', 'NDCG', 'MRR', 'Hit',
                          'ItemCoverage', 'GiniIndex',
                          'AveragePopularity',
                          'TierExposure', 'ItemCoverageN']
config['topk']         = [10, 20]
config['valid_metric'] = 'NDCG@10'
config['eval_args']    = {'mode': 'full'}
config['tail_ratio']   = 0.2

L = build_lookups(dataset, config['dataset'])
config['popularity_pool'] = list(L['pop_set'])
config['mid_pool']        = list(L['mid_set'])
config['niche_pool']      = list(L['niche_set'])

# ── patch once, up front ────────────────────────────────────────
cbm_sasrec.steer_concept_idx = None
cbm_sasrec.steer_scale = 1.0
cbm_sasrec.forward = types.MethodType(patched_forward, cbm_sasrec)

concept_idx_pop = cbm_sasrec.concept_names.index('popularity')
concept_idx_mid = cbm_sasrec.concept_names.index('niche')

# ── baseline ────────────────────────────────────────────────────
cbm_sasrec._interventions = {}
result_baseline = evaluate_with_recbole(cbm_sasrec, config, train_data, test_data)

# ── sweep grid ──────────────────────────────────────────────────
# ── sweep grid ──────────────────────────────────────────────────
#alphas = np.round(np.arange(1.0, -2, -0.05), 2)   # popularity scale: 1.0 → 0.0
#betas  = np.round(np.arange(4.0,  0.9, -0.05), 2)   # mid scale:        3.0 → 1.0
alphas = np.round(np.arange(0, -3, -0.1), 2)   # popularity scale: 1.0 → 0.0
betas  = np.round(np.arange(0, 5, 0.1), 2)   # mid scale:        3.0 → 1.0

rows = []
rows.append({'alpha': 1.0, 'beta': 1.0, 'is_baseline': True, **result_baseline})

gap_before = result_baseline['pop_rate@10'] - result_baseline['mid_rate@10']

for alpha in alphas:                       # outer loop: popularity
    for beta in betas:                     # inner loop: mid
        

        cbm_sasrec._interventions = {
            concept_idx_pop: float(alpha),
            concept_idx_mid: float(beta),
        }

        result = evaluate_with_recbole(cbm_sasrec, config, train_data, test_data)
        rows.append({'alpha': float(alpha), 'beta': float(beta),
                     'is_baseline': False, **result})

      

cbm_sasrec._interventions = {}

# ── final summary ───────────────────────────────────────────────
# ── final summary ───────────────────────────────────────────────
df = pd.DataFrame(rows)

# clean column aliases for the metrics you care about (both cutoffs)
metric_map = {
    'ndcg':          'ndcg',
    'hit':           'hit',
    'avg_pop':       'averagepopularity',
    'item_cov':      'itemcoverage',
    'item_cov_n':    'itemcoveragen',   # check exact key, see note below
    'gini':          'giniindex',
}



out_path = f"./dataset/{dataset_name}/results/SASRec_CBM_sigmoid_additive_before_sigmoid_interv_tmp_{dataset_name}-results.csv"
df.to_csv(out_path, index=False)
print(f"\nSaved {len(df)} rows to {out_path}")



  pop_set:   1,370 items (≥ 220 interactions)
  mid_set:   4,059 items
  niche_set: 1,418 items (1–47 interactions)
  Items in metadata: 6,849, matched to internal IDs: 6,849
  N_CONCEPTS = 22  (16 genres + 6 scalars)
c_hat_after_pop: tensor(0.4979, device='cuda:0')
c_hat_after_mid: tensor(0.0387, device='cuda:0')
c_hat_after_pop: tensor(0.4910, device='cuda:0')
c_hat_after_mid: tensor(0.0383, device='cuda:0')
c_hat_after_pop: tensor(0.4975, device='cuda:0')
c_hat_after_mid: tensor(0.0369, device='cuda:0')
c_hat_after_pop: tensor(0.5117, device='cuda:0')
c_hat_after_mid: tensor(0.0356, device='cuda:0')
c_hat_after_pop: tensor(0.5126, device='cuda:0')
c_hat_after_mid: tensor(0.0354, device='cuda:0')
c_hat_after_pop: tensor(0.5154, device='cuda:0')
c_hat_after_mid: tensor(0.0336, device='cuda:0')
c_hat_after_pop: tensor(0.4974, device='cuda:0')
c_hat_after_mid: tensor(0.0345, device='cuda:0')
c_hat_after_pop: tensor(0.4979, device='cuda:0')
c_hat_after_mid: tensor(0.0387, device='cuda:0'

In [ ]:
len(L['pop_set'])

In [34]:
tmp=pd.read_csv("./dataset/lastfm/results/SASRec_CBM_sigmoid_additive_before_sigmoid_interv_tmp_lastfm-results.csv")
tmp[tmp['ndcg@20']>0.1372]['averagepopularity@20'].min()

160.5168

In [32]:
lsfm=pd.read_csv("/home/mvarasteh/post-hoc/dataset/lastfm/results/SASRec_SAE_popsteer_lastfm-results.csv")
lsfm[lsfm['ndcg@20']>0.1372]['avgpop@20'].min()

84.1162

In [ ]:
import types
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from recbole.utils.build_concepts import build_lookups
from recbole.evaluator import Collector, Evaluator
from recbole.trainer import Trainer

dataset_name = "lastfm"

# ═══════════════════════════════════════════════════════════════
# Patched forward: conditional mass-transfer intervention (eval only)
#   _transfer = {src, dst, thresh, factor}
#     for users with c_hat[src] > thresh:
#       remove factor * c_hat[src] from src, add it to dst
#   src/dst must be in the SAME softmax group (simplex preserved).
#   Set _transfer = None for baseline.
# ═══════════════════════════════════════════════════════════════
def patched_forward(self, item_seq, item_seq_len):
    h = self._encode(item_seq, item_seq_len)

    #logits = self.concept_predictor(h)
    logits = self.concept_predictor(h)

    z = self.latent_predictor(h)

    # group-wise softmax — ALWAYS runs
    c_hat = torch.cat([
        F.softmax(logits[:, :16],   dim=-1),   # genres
        F.softmax(logits[:, 16:19], dim=-1),   # popularity tiers
        F.softmax(logits[:, 19:22], dim=-1),   # era bins
    ], dim=-1)

    # Mode 2: conditional mass transfer (post-softmax, simplex-preserving)
    if getattr(self, '_transfer', None) and not self.training:
        t = self._transfer
        c_hat = c_hat.clone()
        mask = c_hat[:, t['src']] > t['thresh']
        removed = c_hat[mask, t['src']] * t['factor']
        c_hat[mask, t['src']] -= removed
        c_hat[mask, t['dst']] += removed
        self._n_affected += mask.sum().item()
        self._n_total    += mask.numel()

    bottleneck = torch.cat([c_hat, z], dim=-1)
    h_hat = self.reconstructor(bottleneck)
    return h, c_hat, z, h_hat


# ═══════════════════════════════════════════════════════════════
# Single eval function (identical path for baseline and steered)
# ═══════════════════════════════════════════════════════════════
def evaluate_with_recbole(model, config, train_data, test_data):
    # reset per-run state
    model._n_affected, model._n_total = 0, 0
    model._eval_c_hat_buffer, model._eval_gt_buffer = [], []

    trainer = Trainer(config, model)
    trainer.eval_collector = Collector(config)
    trainer.evaluator      = Evaluator(config)
    trainer.eval_collector.data_collect(train_data)

    result = trainer.evaluate(test_data, load_best_model=False, show_progress=False)

    result['affected_frac'] = (model._n_affected / model._n_total
                               if model._n_total > 0 else 0.0)
    model._eval_c_hat_buffer, model._eval_gt_buffer = [], []
    return result


# ═══════════════════════════════════════════════════════════════
# Config
# ═══════════════════════════════════════════════════════════════
config['metrics']      = ['Recall', 'NDCG', 'MRR', 'Hit',
                          'ItemCoverage', 'GiniIndex',
                          'AveragePopularity',
                          'TierExposure', 'ItemCoverageN']
config['topk']         = [10, 20]
config['valid_metric'] = 'NDCG@10'
config['eval_args']    = {'mode': 'full'}
config['tail_ratio']   = 0.2

L = build_lookups(dataset, config['dataset'])
config['popularity_pool'] = list(L['pop_set'])
config['mid_pool']        = list(L['mid_set'])
config['niche_pool']      = list(L['niche_set'])

# ═══════════════════════════════════════════════════════════════
# Patch model BEFORE any eval
# ═══════════════════════════════════════════════════════════════
cbm_sasrec.forward = types.MethodType(patched_forward, cbm_sasrec)
cbm_sasrec._transfer = None
cbm_sasrec._n_affected, cbm_sasrec._n_total = 0, 0

concept_idx_src = cbm_sasrec.concept_names.index('popularity')
concept_idx_dst = cbm_sasrec.concept_names.index('mid')
# both must be inside the pop-tier softmax group [16, 19)
assert 16 <= concept_idx_src < 19 and 16 <= concept_idx_dst < 19, \
    f"src={concept_idx_src}, dst={concept_idx_dst} must be in the same group"

# ═══════════════════════════════════════════════════════════════
# Baseline (no transfer)
# ═══════════════════════════════════════════════════════════════
cbm_sasrec._transfer = None
result_baseline = evaluate_with_recbole(cbm_sasrec, config, train_data, test_data)

gap_before = result_baseline['pop_rate@10'] - result_baseline['mid_rate@10']
print(f"baseline  ndcg@10={result_baseline['ndcg@10']:.4f}  "
      f"pop-mid gap={gap_before:+.4f}", flush=True)

# ═══════════════════════════════════════════════════════════════
# Sweep grid — mode 2 knobs
#   thresh: only users with predicted pop prob > thresh are touched
#           (0.0 = unconditional; uniform in a 3-way group is 0.33)
#   factor: fraction of pop's mass moved to mid
# ═══════════════════════════════════════════════════════════════
threshs = [0.0, 0.1,0.15, 0.2, 0.25, 0.3, 0.35, 0.4,0.45, 0.5,0.55, 0.6, 0.65, 0.7]
factors = [0.1,0.15,  0.2,0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.75, 1.0, 1.2, 1.3, 1.5 ,2]

out_path = f"./dataset/{dataset_name}/results/SASRec_CBM_softmax_transfer_tmp_{dataset_name}-results.csv"

def clean_columns(df):
    df = df.copy()
    df.columns = [c.replace('averagepopularity', 'avgpop')
                   .replace('giniindex', 'gini')
                  for c in df.columns]
    return df

rows = [{'thresh': np.nan, 'factor': 0.0, 'is_baseline': True, **result_baseline}]

n_total = len(threshs) * len(factors)
n_done = 0

for thresh in threshs:
    for factor in factors:
        cbm_sasrec._transfer = {
            'src':    concept_idx_src,
            'dst':    concept_idx_dst,
            'thresh': float(thresh),
            'factor': float(factor),
        }
        result = evaluate_with_recbole(cbm_sasrec, config, train_data, test_data)
        rows.append({'thresh': float(thresh), 'factor': float(factor),
                     'is_baseline': False, **result})

        n_done += 1
        gap = result['pop_rate@10'] - result['mid_rate@10']
        print(f"[{n_done}/{n_total}] thresh={thresh:.2f} factor={factor:.2f}  "
              f"ndcg@10={result['ndcg@10']:.4f}  gap={gap:+.4f}  "
              f"affected={result['affected_frac']:.1%}", flush=True)

        # incremental save — a crash mid-sweep loses nothing
        clean_columns(pd.DataFrame(rows)).to_csv(out_path, index=False)

cbm_sasrec._transfer = None

# ═══════════════════════════════════════════════════════════════
# Final save + quick summary
# ═══════════════════════════════════════════════════════════════
df = clean_columns(pd.DataFrame(rows))
df.to_csv(out_path, index=False)
print(f"\nSaved {len(df)} rows to {out_path}")

floor = 0.97 * result_baseline['ndcg@10']
valid = df[(~df['is_baseline']) & (df['ndcg@10'] >= floor)]
if not valid.empty:
    valid = valid.assign(gap=valid['pop_rate@10'] - valid['mid_rate@10'])
    best = valid.loc[valid['gap'].abs().idxmin()]
    print(f"best config within 3% NDCG budget: thresh={best['thresh']:.2f} "
          f"factor={best['factor']:.2f}  gap={best['gap']:+.4f} "
          f"(baseline {gap_before:+.4f})  ndcg@10={best['ndcg@10']:.4f}  "
          f"affected={best['affected_frac']:.1%}")
else:
    print("no config stays within the 3% NDCG budget — grid may be too aggressive")

## Creating Table-Accuracy-NDCG trade off

In [28]:
import pandas as pd
from pathlib import Path

# ---------------- config ----------------
datasets = ["lastfm"]
#models   = ["DUOR", "FAIR", "IPR", "PCT", "PMMF","PopSteer","ConcRec", "SAE_popsteer", "CBM"]  # add your 7th
models   = ["SAE_popsteer"]  # add your 7th

thresholds = [0.03, 0.06, 0.09]

# frozen SASRec NDCG per dataset — fill in your actual values
base_ndcg = {
    "ml-1mm": 0.1415,#0.1679
    "lastfm": 0.2087,
    "beeradvocate":  0.1099,
}

#results_dir = Path("/home/mvarasteh/post-hoc/dataset/beeradvocate/results")   # adjust
select_by = "cov"   # "cov" -> row with max coverage; "gini" -> row with min gini
acc_metric="ndcg@10"

fair_met="covn@10"
sec_metric="gini@10"

# ---------------- main ----------------
records = []
for ds in datasets:
    results_dir = Path(f"/home/mvarasteh/post-hoc/dataset/{ds}/results")   # adjust

    for model in models:
        # handles both flat folder and per-dataset subfolders
        matches = list(results_dir.rglob(f"SASRec_{model}_{ds}-results.csv"))
        if not matches:
            print(f"missing: SASRec_{model}_{ds}-results.csv")
            continue
        df = pd.read_csv(matches[0])

        for t in thresholds:
            floor = (1 - t) * base_ndcg[ds]
            valid = df[df[acc_metric] >= floor]
            print(floor)
            if valid.empty:
                rec = {"cov": None, "gini": None, acc_metric: None, "n_valid": 0}
            else:
                if select_by == "cov":
                    row = valid.loc[valid[fair_met].idxmax()]
                else:
                    row = valid.loc[valid[sec_metric].idxmin()]
                rec = {
                    fair_met:  row[fair_met],
                    sec_metric: row[sec_metric],
                    acc_metric: row[acc_metric],       # actual NDCG of the chosen config
                    "n_valid": len(valid),
                }


            records.append({
                "dataset": ds, "model": model,
                "threshold": f"{int(t*100)}%", **rec,
            })

summary = pd.DataFrame(records)

# paper-style tables: rows = model, columns = threshold, one table per dataset+metric
for ds in datasets:
    print(summary.columns.tolist())
    sub = summary[summary["dataset"] == ds]
    cov_tab  = sub.pivot(index="model", columns="threshold", values=fair_met)
    gini_tab = sub.pivot(index="model", columns="threshold", values=sec_metric)
    print(f"\n=== {ds} — {acc_metric} ===")
    print(cov_tab.round(4))
    print(f"\n=== {ds} — {sec_metric} ===")
    print(gini_tab.round(4))

#summary.to_csv("Table_summary.csv", index=False)

KeyError: 'ndcg@10'

In [ ]:
pop_steer=pd.read_csv("/home/mvarasteh/post-hoc/dataset/ml-1mm/results/SASRec_ConcRec_ml-1mm-results.csv", sep=",")

In [ ]:
pop_steer['cov@20'].max()

## plot

In [ ]:
dataset_name='ml-1mm'
dataset_name_plot="ML-1M"

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from pathlib import Path
import matplotlib.ticker as ticker


def select_best_per_ndcg_bin(df, x_col, y_col, bins, lower_is_better=True):
    """
    For each NDCG bin, pick the row with the best y value.
    lower_is_better=True  → lowest Gini  (use for Gini index)
    lower_is_better=False → highest value (use for Item Coverage)
    """
    records = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (df[x_col] >= lo) & (df[x_col] < hi)
        subset = df[mask]
        if len(subset) == 0:
            continue
        if lower_is_better:
            best = subset.loc[subset[y_col].idxmin()]
        else:
            best = subset.loc[subset[y_col].idxmax()]
        records.append(best)
    return pd.DataFrame(records)


def make_label(path, dataset_name):
    name = path.stem
    name = name.replace(f'_{dataset_name}-results', '').replace('SASRec_', '')
    name = name.replace('CBM', 'ConceptRec')
    return name


def plot_ndcg_vs_metric(
    dataset_name,
    dataset_name_plot,
    y_col='gini@10',
    y_label='Gini@10',
    lower_is_better=True,
    ndcg_bins=None,
    xlim=(0.09, 0.155),
    ylim=None,
    x_tick_step=0.02,
    y_tick_step=0.05,
    n_samples=10,
    save_path=None,
):
    results_dir = Path(f'./dataset/{dataset_name}/results')
    csv_files = sorted(results_dir.glob('*.csv'))

    if ndcg_bins is None:
        ndcg_bins = np.arange(xlim[0], xlim[1] + x_tick_step, x_tick_step / 2)

    FONT_SIZE = 14
    fig, ax = plt.subplots(figsize=(6, 4))
    colors = cm.tab10(np.linspace(0, 0.9, len(csv_files)))
    markers = ['s', 'o', '^', 'D', 'P', 'X', 'v', '*']

    legend_handles = []
    legend_labels  = []
    legend_seen    = set()

    for i, (csv_path, color) in enumerate(zip(csv_files, colors)):
        df = pd.read_csv(csv_path)

        if 'param1' not in df.columns:
            df['param1'] = 100
        if 'param2' not in df.columns:
            df['param2'] = 100

        label      = make_label(csv_path, dataset_name)
        marker     = markers[i % len(markers)]
        plot_color = 'red' if 'ConcRec' in label else color

        base    = df[(df['param1'] == 100) & (df['param2'] == 100)]
        steered = df[(df['param1'] != 100) | (df['param2'] != 100)]

        # ── Steered points: best y per NDCG bin ──────────────────────────
        binned = select_best_per_ndcg_bin(
            steered, 'ndcg', y_col, ndcg_bins,
            lower_is_better=lower_is_better
        )

        if len(binned) > 0:
            sc = ax.scatter(
                binned['ndcg'], binned[y_col],
                color=plot_color, s=60, marker=marker,
                zorder=2, edgecolors='black', linewidths=0.5,
                label=label,
            )
            if label not in legend_seen:
                legend_handles.append(sc)
                legend_labels.append(label)
                legend_seen.add(label)

        # ── Baseline star ─────────────────────────────────────────────────
        if not base.empty:
            b = ax.scatter(
                base['ndcg'], base[y_col],
                color='black', s=200, marker='*',
                zorder=3, edgecolors='black', linewidths=1.2,
                label='SASRec',
            )
            if 'SASRec' not in legend_seen:
                legend_handles.append(b)
                legend_labels.append('SASRec')
                legend_seen.add('SASRec')

    # ── Axes formatting ───────────────────────────────────────────────────
    ax.set_xlabel('nDCG', fontsize=FONT_SIZE)
    ax.set_ylabel(y_label, fontsize=FONT_SIZE)
    ax.set_title(dataset_name_plot, fontsize=FONT_SIZE + 1)
    ax.tick_params(axis='both', labelsize=FONT_SIZE - 1)

    ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)

    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
    ax.xaxis.set_major_locator(ticker.MultipleLocator(x_tick_step))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(y_tick_step))

    # ── Legend ───
    # ─────────────────────────────────────────────────────────
    ax.legend(
        handles=legend_handles, labels=legend_labels,
        fontsize=FONT_SIZE - 3,
        loc='upper center', bbox_to_anchor=(0.5, -0.16),
        ncol=len(legend_handles), frameon=True,
        handlelength=1.5, handletextpad=0.4, columnspacing=0.9,
        markerscale=1.1,
    )

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300, pad_inches=0.1)
        print(f"Saved to {save_path}")

    plt.show()


# ── Example usage ─────────────────────────────────────────────────────────


 #Item Coverage plot (higher is better)
plot_ndcg_vs_metric(
     dataset_name=dataset_name,
     dataset_name_plot=dataset_name_plot,
     y_col='cov@10',
     y_label='Item Coverage',
     lower_is_better=False,
     ndcg_bins=np.arange(0.01, 0.145, 0.005),
     xlim=(0.08, 0.145),
     x_tick_step=0.1,
     y_tick_step=0.1,
     save_path=f'results_plot/ndcg_vs_cov_{dataset_name}.pdf',
 )




# Gini index plot (lower is better)

plot_ndcg_vs_metric(
    dataset_name=dataset_name,
    dataset_name_plot=dataset_name_plot,
    y_col='gini@10',
    y_label='Gini Index',
    lower_is_better=True,
    ndcg_bins=np.arange(0.001, 0.2, 0.005),
    xlim=(0.08, 0.145),
    x_tick_step=0.05,
    y_tick_step=0.01,
    save_path=f'results_plot/ndcg_vs_gini_{dataset_name}.pdf',
)




## TMP

In [ ]:
import types
import torch
from recbole.evaluator import Collector, Evaluator
from recbole.quick_start import load_data_and_model

# Save the original forward so you can restore it cleanly
if not hasattr(cbm_sasrec, '_original_forward'):
    cbm_sasrec._original_forward = cbm_sasrec.forward

def forward_with_gt(self, item_seq, item_seq_len):
    """Diagnostic forward: bottleneck = [gt_concepts ; z], no interventions."""
    h     = self._encode(item_seq, item_seq_len)
    c_hat = self.concept_predictor(h)         # still computed, returned for buffers
    z     = self.latent_predictor(h)
    gt    = self._lookup_concepts(item_seq)

    bottleneck = torch.cat([gt, z], dim=-1)   # ← gt replaces c_hat
    h_hat      = self.reconstructor(bottleneck)
    return h, c_hat, z, h_hat
    #return h, c_hat, h_hat
# ── Baseline: standard c_hat path ────────────────────────────────────────
cbm_sasrec._interventions = {}
cbm_sasrec.forward = cbm_sasrec._original_forward    # ensure original
result_chat = evaluate_with_recbole_sasrec(cbm_sasrec, config, train_data, test_data)

# ── Diagnostic: gt path ──────────────────────────────────────────────────
cbm_sasrec.forward = types.MethodType(forward_with_gt, cbm_sasrec)
try:
    result_gt = evaluate_with_recbole_sasrec(cbm_sasrec, config, train_data, test_data)
finally:
    cbm_sasrec.forward = cbm_sasrec._original_forward    # always restore

# ── Compare ──────────────────────────────────────────────────────────────
print(f"\n{'Metric':<30} {'c_hat path':>12} {'gt path':>12} {'Δ':>12}  {'Δ%':>8}")
print("─" * 80)
for k in sorted(result_chat.keys()):
    b, s = result_chat[k], result_gt[k]
    delta = s - b
    pct = (100 * delta / b) if abs(b) > 1e-9 else 0.0
    print(f"{k:<30} {b:>12.4f} {s:>12.4f} {delta:>+12.4f}  {pct:>+7.2f}%")

## Steering as Counterfactual Explanation

In [ ]:
"""
Individual-user concept steering.

For a single user:
  1. Run the CBM, get c_hat (predicted concept activations).
  2. Pick the top-3 concepts by c_hat value.
  3. Suppress them (set to 0) — both individually and all-3-together.
  4. Re-generate top-20 recommendations from the suppressed c_hat.
  5. Measure how much the top-20 list changes (RBO / Jaccard / overlap),
     and which concepts moved.

This bypasses cbm_model.forward()'s built-in steering hook so the only
intervention applied is the one we control here.
"""



torch.manual_seed(42)
@torch.no_grad()
def _encode_user(cbm_model, interaction, device):
    """Return h, c_hat, z (UN-steered) for a one-row interaction."""
    item_seq     = interaction[cbm_model.ITEM_SEQ]
    item_seq_len = interaction[cbm_model.ITEM_SEQ_LEN]
    h     = cbm_model._encode(item_seq, item_seq_len)
    c_hat = cbm_model.concept_predictor(h)
    z     = cbm_model.latent_predictor(h)         # ← NEW
    return h, c_hat, z, item_seq, item_seq_len    # ← 5-tuple now
    #return h, c_hat, item_seq, item_seq_len

def _scores_from_chat(cbm_model, c_hat,z):
    """[c_hat ; z] -> h_hat -> full item scores [1, n_items]."""
    bottleneck = torch.cat([c_hat, z], dim=-1)    # ← concat to 41 dims
    h_hat = cbm_model.reconstructor(bottleneck)
    return h_hat @ cbm_model.item_embedding.weight.T


@torch.no_grad()
def steer_single_user(cbm_model, interaction, k=20, top_n=3,
                       use_ground_truth=False, device=None,
                       exclude_seen=True, verbose=True,
                       steer_target='concept'):   # ← NEW: 'concept' or 'latent'
    if device is None:
        device = next(cbm_model.parameters()).device

    cbm_model.eval()
    interaction = interaction.to(device)

    h, c_hat, z, item_seq, item_seq_len = _encode_user(cbm_model, interaction, device)
    c_hat = c_hat.clone()
    z     = z.clone()                             # ← clone z too

    concept_names = getattr(cbm_model, 'concept_names', None)
    def name(i):
        return concept_names[i] if concept_names is not None else f"concept_{i}"

    # ── choose steering vector and its label ─────────────────────────────────
    if steer_target == 'latent':
        steering_vec = z[0].clone()               # [16]
        steer_label  = "latent (z)"
    else:
        if use_ground_truth:
            steering_vec = cbm_model._lookup_concepts(item_seq).clone()[0]
            steer_label  = "ground-truth concept"
        else:
            steering_vec = c_hat[0].clone()
            steer_label  = "c_hat concept"

    #top_idx = torch.topk(steering_vec, top_n, largest=True).indices.tolist()
    #top_idx = torch.topk(steering_vec, top_n, largest=False).indices.tolist()

    top_idx = torch.randperm(steering_vec.shape[0])[:top_n].tolist()

    # ── seen-item mask ────────────────────────────────────────────────────────
    seen = set()
    if exclude_seen:
        seen = set(item_seq[0].detach().cpu().tolist()) - {0}

    def topk_from_scores(scores):
        s = scores.clone()
        if seen:
            s[0, list(seen)] = float('-inf')
        s[0, 0] = float('-inf')
        return torch.topk(s[0], k).indices.detach().cpu().tolist()

    # ── baseline ──────────────────────────────────────────────────────────────
    base_scores = _scores_from_chat(cbm_model, c_hat, z)
    base_list   = topk_from_scores(base_scores)

    if verbose:
        print(f"\n{'='*64}")
        print(f"Single-user steering  (target: {steer_label})")
        print(f"{'='*64}")
        print(f"  Top-{top_n} dims to suppress (by smallest activation):")
        for i in top_idx:
            val = steering_vec[i].item()
            print(f"    [{i:3d}]  {'z' if steer_target=='latent' else name(i):<18s}  val={val:.4f}")

    def compare(new_list):
        sa, sb = set(base_list), set(new_list)
        inter = len(sa & sb)
        union = len(sa | sb)
        return {
            'list':    new_list,
            'rbo':     rbo.RankingSimilarity(base_list, new_list).rbo(),
            'jaccard': inter / union if union else 0.0,
            'overlap': inter,
            'changed': k - inter,
        }

    results = {
        'steer_target':  steer_target,
        'top_dims':      [(i, ('z' if steer_target == 'latent' else name(i)),
                           float(steering_vec[i])) for i in top_idx],
        'baseline_list': base_list,
        'individual':    {},
        'combined':      None,
    }

    # ── individual suppression ────────────────────────────────────────────────
    if verbose:
        print(f"\n  Individual suppression (one dim -> 0):")
    for i in top_idx:
        if steer_target == 'latent':
            c_mod, z_mod = c_hat.clone(), z.clone()
            z_mod[0, i] = 0                       # ← zero in z
        else:
            c_mod, z_mod = c_hat.clone(), z.clone()
            c_mod[0, i] = 0                       # ← zero in c_hat

        r = compare(topk_from_scores(_scores_from_chat(cbm_model, c_mod, z_mod)))
        results['individual'][i] = r
        if verbose:
            label = f"z[{i}]" if steer_target == 'latent' else name(i)
            print(f"    {label:<18s}  "
                  f"RBO={r['rbo']:.4f}  Jaccard={r['jaccard']:.4f}  "
                  f"overlap={r['overlap']:2d}/{k}  changed={r['changed']:2d}")

    # ── combined suppression ──────────────────────────────────────────────────
    c_mod, z_mod = c_hat.clone(), z.clone()
    for i in top_idx:
        if steer_target == 'latent':
            z_mod[0, i] = 0
        else:
            c_mod[0, i] = 0

    rc = compare(topk_from_scores(_scores_from_chat(cbm_model, c_mod, z_mod)))
    results['combined'] = rc
    if verbose:
        print(f"\n  Combined suppression (all {top_n} -> 0):")
        print(f"    RBO={rc['rbo']:.4f}  Jaccard={rc['jaccard']:.4f}  "
              f"overlap={rc['overlap']:2d}/{k}  changed={rc['changed']:2d}")
        print(f"{'='*64}\n")

    return results











@torch.no_grad()
def steer_population(cbm_model, data_loader, k=20, top_n=3,
                     use_ground_truth=False, max_users=None, device=None,
                     steer_target='concept'):                # ← NEW
    
    """
    Run steer_single_user over many users and aggregate.
    Iterates the loader row-by-row so each user is steered by *their own*
    top concepts. Returns arrays of combined-suppression metrics.
    """

    if device is None:
        device = next(cbm_model.parameters()).device
    cbm_model.eval()

    rbo_all, jac_all, chg_all = [], [], []
    n_done = 0

    for batch in data_loader:
        interaction = batch[0] if isinstance(batch, (list, tuple)) else batch
        interaction = interaction.to(device)
        B = interaction[cbm_model.ITEM_SEQ].shape[0]

        for b in range(B):
            row = interaction[b:b + 1]
            r = steer_single_user(
                cbm_model, row, k=k, top_n=top_n,
                use_ground_truth=use_ground_truth,
                device=device, verbose=False,
                steer_target=steer_target
            )
            c = r['combined']
            rbo_all.append(c['rbo'])
            jac_all.append(c['jaccard'])
            chg_all.append(c['changed'])
            n_done += 1
            if max_users is not None and n_done >= max_users:
                break
        if max_users is not None and n_done >= max_users:
            break

    rbo_all = np.array(rbo_all)
    jac_all = np.array(jac_all)
    chg_all = np.array(chg_all)

    print(f"\n{'='*64}")
    print(f"Population steering summary  "
          f"(top-{top_n} suppressed, combined, N={n_done})")
    print(f"{'='*64}")
    print(f"  Mean RBO vs baseline:    {rbo_all.mean():.4f}  "
          f"(1.0 = no change, lower = bigger change)")
    print(f"  Median RBO:              {np.median(rbo_all):.4f}")
    print(f"  Mean Jaccard:            {jac_all.mean():.4f}")
    print(f"  Mean items changed /{k}:  {chg_all.mean():.2f}")
    print(f"  Users w/ >=1 change:     {(chg_all > 0).sum()} / {n_done} "
          f"({100*(chg_all>0).mean():.1f}%)")
    print(f"  Users w/ identical list: {(chg_all == 0).sum()} / {n_done}")
    print(f"{'='*64}\n")

    return {'rbo': rbo_all, 'jaccard': jac_all, 'changed': chg_all}

In [ ]:
# ---- single user ----
import numpy as np
batch = next(iter(test_data))
interaction = batch[0] if isinstance(batch, (list, tuple)) else batch
one_user = interaction[234:235]

res = steer_single_user(cbm_sasrec, one_user, k=20, top_n=3,
                        use_ground_truth=False)   # top-3 by c_hat

combined = res['combined']     # <-- this is your "all 3 at once" result
print(combined['rbo'], combined['jaccard'], combined['changed'])

In [ ]:
import numpy as np

results_concept=steer_population(cbm_sasrec, test_data, k=20, top_n=3,
                     use_ground_truth=False, max_users=None, device=None,steer_target='concept')

results_latent=steer_population(cbm_sasrec, test_data, k=20, top_n=3,
                     use_ground_truth=False, max_users=None, device=None,steer_target='latent')

In [ ]:
from sklearn.metrics import jaccard_score
y_true = np.array([[0.9, 1, 1],
                   [1, 1, 0]])
y_pred = np.array([[1, 1, 1],
                   [1, 0, 0]])
jaccard_score(y_true[0], y_pred[0])

In [ ]:
"""
Intervenability evaluation for CBM-based sequential recommender.

Tests: does replacing predicted concept values with ground-truth values
improve held-out predictive performance? (Margeloiu et al. 2021)

Reports HR@k, NDCG@k, and Item Coverage before/after intervention.
"""

import numpy as np
import torch


@torch.no_grad()
def evaluate_intervenability(
    cbm_model,
    eval_loader,
    k=20,
    selection='error',           # 'error' | 'predicted' | 'random' | 'all'
    n_intervene=3,               # how many concepts to replace with gt (ignored if selection='all')
    exclude_seen=True,
    max_users=None,
    device=None,
    verbose=True,
):
    """
    Args:
        cbm_model     : trained SASRec_CBM in eval mode.
        eval_loader   : RecBole full-sort eval dataloader (must yield ITEM_ID target).
        k             : top-K for HR/NDCG/coverage.
        selection     : which concepts to replace with ground truth:
                          'error'     -> top-n by |c_hat - gt|  (canonical intervenability test)
                          'predicted' -> top-n by c_hat value
                          'random'    -> n random concepts
                          'all'       -> replace ALL concepts (concept oracle)
        n_intervene   : number of concepts to intervene on (ignored if selection='all').
        exclude_seen  : mask items already in the user's history.
        max_users     : cap number of users (None = full eval set).
    Returns:
        dict with before/after metrics, deltas, and per-user arrays.
    """
    if device is None:
        device = next(cbm_model.parameters()).device
    cbm_model.eval()

    n_items    = cbm_model.item_embedding.weight.shape[0]
    n_concepts = cbm_model.n_concepts

    # accumulators
    hr_before,   hr_after   = [], []
    ndcg_before, ndcg_after = [], []
    items_recommended_before = set()
    items_recommended_after  = set()
    target_concept_idx = []     # which concepts were intervened on per user (for stratification)
    n_done = 0

    def topk_with_mask(c_vec, seen):
        """c_vec: [1, n_concepts] -> top-k item indices (list)."""
        h_hat  = cbm_model.reconstructor(c_vec)
        scores = (h_hat @ cbm_model.item_embedding.weight.T)[0]
        if seen:
            scores[list(seen)] = float('-inf')
        scores[0] = float('-inf')   # padding item
        return torch.topk(scores, k).indices.cpu().tolist()

    def hr_score(topk_list, target):
        return float(target in topk_list)

    def ndcg_score(topk_list, target):
        if target in topk_list:
            return 1.0 / np.log2(topk_list.index(target) + 2)
        return 0.0

    for batch in eval_loader:
        interaction = (batch[0] if isinstance(batch, (list, tuple)) else batch).to(device)
        B = interaction[cbm_model.ITEM_SEQ].shape[0]

        for b in range(B):
            row          = interaction[b:b + 1]
            item_seq     = row[cbm_model.ITEM_SEQ]
            item_seq_len = row[cbm_model.ITEM_SEQ_LEN]
            target       = row[cbm_model.ITEM_ID].item()

            # --- encode user, get predicted and gt concepts ---
            h     = cbm_model._encode(item_seq, item_seq_len)
            c_hat = cbm_model.concept_predictor(h).clone()        # [1, n_concepts]
            gt    = cbm_model._lookup_concepts(item_seq).clone()  # [1, n_concepts]

            # --- choose which concepts to replace ---
            if selection == 'all':
                idx_to_replace = list(range(n_concepts))
            else:
                if selection == 'error':
                    order = (c_hat[0] - gt[0]).abs().argsort(descending=True)
                elif selection == 'predicted':
                    order = c_hat[0].argsort(descending=True)
                elif selection == 'random':
                    order = torch.randperm(n_concepts, device=device)
                else:
                    raise ValueError(f"unknown selection: {selection}")
                idx_to_replace = order[:n_intervene].tolist()

            target_concept_idx.append(idx_to_replace)

            # --- build intervened concept vector ---
            c_mod = c_hat.clone()
            for i in idx_to_replace:
                c_mod[0, i] = gt[0, i]

            # --- seen-item mask ---
            seen = set(item_seq[0].cpu().tolist()) - {0} if exclude_seen else set()

            # --- score before and after ---
            topk_before = topk_with_mask(c_hat, seen)
            topk_after  = topk_with_mask(c_mod, seen)

            # --- metrics ---
            hr_before.append(hr_score(topk_before, target))
            hr_after.append(hr_score(topk_after,  target))
            ndcg_before.append(ndcg_score(topk_before, target))
            ndcg_after.append(ndcg_score(topk_after,  target))

            items_recommended_before.update(topk_before)
            items_recommended_after.update(topk_after)

            n_done += 1
            if max_users is not None and n_done >= max_users:
                break
        if max_users is not None and n_done >= max_users:
            break

    # --- aggregate ---
    hr_b,   hr_a   = np.array(hr_before),   np.array(hr_after)
    ndcg_b, ndcg_a = np.array(ndcg_before), np.array(ndcg_after)

    results = {
        'n_users':         n_done,
        'selection':       selection,
        'n_intervene':     n_intervene if selection != 'all' else n_concepts,
        'k':               k,

        f'hr@{k}_before':       hr_b.mean(),
        f'hr@{k}_after':        hr_a.mean(),
        f'hr@{k}_delta':        hr_a.mean() - hr_b.mean(),

        f'ndcg@{k}_before':     ndcg_b.mean(),
        f'ndcg@{k}_after':      ndcg_a.mean(),
        f'ndcg@{k}_delta':      ndcg_a.mean() - ndcg_b.mean(),

        f'coverage@{k}_before': len(items_recommended_before) / n_items,
        f'coverage@{k}_after':  len(items_recommended_after)  / n_items,
        f'coverage@{k}_delta':  (len(items_recommended_after) -
                                 len(items_recommended_before)) / n_items,

        # per-user arrays for further analysis / significance testing
        '_per_user': {
            'hr_before':   hr_b,
            'hr_after':    hr_a,
            'ndcg_before': ndcg_b,
            'ndcg_after':  ndcg_a,
            'intervened_concepts': target_concept_idx,
        },
    }

    if verbose:
        print(f"\n{'='*72}")
        print(f"Intervenability evaluation")
        print(f"  selection      : {selection}")
        print(f"  n_intervene    : {results['n_intervene']}")
        print(f"  users          : {n_done}")
        print(f"  top-k          : {k}")
        print(f"{'='*72}")
        print(f"  {'Metric':<20s}  {'Before':>10s}  {'After':>10s}  {'Δ':>10s}")
        print(f"  {'-'*54}")
        print(f"  {'HR@'+str(k):<20s}  "
              f"{results[f'hr@{k}_before']:>10.4f}  "
              f"{results[f'hr@{k}_after']:>10.4f}  "
              f"{results[f'hr@{k}_delta']:>+10.4f}")
        print(f"  {'NDCG@'+str(k):<20s}  "
              f"{results[f'ndcg@{k}_before']:>10.4f}  "
              f"{results[f'ndcg@{k}_after']:>10.4f}  "
              f"{results[f'ndcg@{k}_delta']:>+10.4f}")
        print(f"  {'Coverage@'+str(k):<20s}  "
              f"{results[f'coverage@{k}_before']:>10.4f}  "
              f"{results[f'coverage@{k}_after']:>10.4f}  "
              f"{results[f'coverage@{k}_delta']:>+10.4f}")
        print(f"{'='*72}\n")

        # paired t-test for statistical significance
        try:
            from scipy.stats import ttest_rel
            t_hr, p_hr     = ttest_rel(hr_a,   hr_b)
            t_ndcg, p_ndcg = ttest_rel(ndcg_a, ndcg_b)
            print(f"  Paired t-test (after vs before):")
            print(f"    HR@{k}:    t={t_hr:>+7.3f}  p={p_hr:.4f}")
            print(f"    NDCG@{k}:  t={t_ndcg:>+7.3f}  p={p_ndcg:.4f}")
            print(f"{'='*72}\n")
        except ImportError:
            pass

    return results



# Replace the 3 most-wrong concepts with ground truth (canonical test)
res_error = evaluate_intervenability(
    cbm_sasrec, test_data,
    k=20, selection='error', n_intervene=3,
)

# Concept oracle: replace ALL concepts with ground truth (upper bound)
res_oracle = evaluate_intervenability(
    cbm_sasrec, test_data,
    k=20, selection='all',
)

# Random baseline: replace 3 random concepts (should help less than 'error')
res_random = evaluate_intervenability(
    cbm_sasrec, test_data,
    k=20, selection='random', n_intervene=3,
)

## steering for individual users

In [ ]:
@torch.no_grad()
def steer_user_concept_exposure(cbm_model, interaction, concept_idx,
                                 k=20, top_changes=10, device=None,
                                 exclude_seen=True, verbose=True):
    """
    Suppress a SPECIFIED concept for one user and report:
      - top-k list before vs after (RBO / Jaccard / overlap)
      - mean concept exposure across the top-k items, before vs after

    interaction  : single-row RecBole Interaction (one user).
    concept_idx  : int or list[int] — concept(s) to suppress to 0.
    k            : recommendation list length.
    top_changes  : how many concepts to display in the exposure table,
                   ranked by |Δ exposure|. The suppressed concept(s) are
                   always shown regardless of rank.

    Returns a dict with the lists, list-change metrics, and the full
    per-concept exposure arrays (baseline, steered, delta).
    """
    
    if device is None:
        device = next(cbm_model.parameters()).device

    cbm_model.eval()
    interaction = interaction.to(device)

    # normalize concept_idx to a list
    if isinstance(concept_idx, int):
        suppressed = [concept_idx]
    else:
        suppressed = list(concept_idx)

    # encode user -> c_hat (un-steered)
    h, c_hat, item_seq, item_seq_len = _encode_user(cbm_model, interaction, device)
    c_hat = c_hat.clone()
    n_concepts = c_hat.shape[1]

    concept_names = getattr(cbm_model, 'concept_names', None)
    def name(i):
        return concept_names[i] if concept_names is not None else f"concept_{i}"

    # seen-item mask + padding
    seen = set()
    if exclude_seen:
        seen = set(item_seq[0].detach().cpu().tolist()) - {0}

    def topk_from_scores(scores):
        s = scores.clone()
        if seen:
            s[0, list(seen)] = float('-inf')
        s[0, 0] = float('-inf')
        return torch.topk(s[0], k).indices.detach().cpu().tolist()

    def exposure_for_items(item_ids):
        """Mean ground-truth concept vector across the given item IDs. -> [n_concepts]"""
        ids = torch.tensor(item_ids, device=device).unsqueeze(0)   # [1, k]
        c_items = cbm_model._lookup_concepts(ids)
        #print(f"item_ids {ids.shape}")
        
        if c_items.dim() == 3:
            # [batch, seq_len, n_concepts] -> mean over seq_len
            return c_items[0].mean(dim=0)                          # [n_concepts]
        elif c_items.dim() == 2:
            # [batch, n_concepts] — already aggregated. Fall back to
            # looking up each item individually so we can average ourselves.
            per_item = torch.stack([
                cbm_model._lookup_concepts(
                    torch.tensor([[iid]], device=device)
                ).squeeze(0).squeeze(0)                            # -> [n_concepts]
                for iid in item_ids
            ])    
            return per_item.mean(dim=0) 
            #return c_items.squeeze(0)                          # [n_concepts]
        else:
            raise RuntimeError(f"Unexpected _lookup_concepts shape: {tuple(c_items.shape)}")
    # ── baseline ─────────────────────────────────────────────────────────────
    base_scores = _scores_from_chat(cbm_model, c_hat)
    base_list   = topk_from_scores(base_scores)
    print(f"base_list{base_list}")
    ids = torch.tensor(base_list, device=device).unsqueeze(0)
    print(f"base_list_concept {cbm_sasrec._lookup_concepts(ids)}")
    base_exp    = exposure_for_items(base_list)                    # [n_concepts]

    # ── steered ──────────────────────────────────────────────────────────────
    c_mod = c_hat.clone()
    for i in suppressed:
        #print(f"Suppressing concept {i} {c_mod}")
        c_mod[0, i] = 0
        #print(f"supresed concept {i} {c_mod}")


    steer_scores = _scores_from_chat(cbm_model, c_mod)
    steer_list   = topk_from_scores(steer_scores)
    steer_exp    = exposure_for_items(steer_list)

    # ── list-change metrics ──────────────────────────────────────────────────
    sa, sb = set(base_list), set(steer_list)
    inter, union = len(sa & sb), len(sa | sb)
    list_metrics = {
        'rbo':     rbo.RankingSimilarity(base_list, steer_list).rbo(),
        'jaccard': inter / union if union else 0.0,
        'overlap': inter,
        'changed': k - inter,
    }

    # ── per-concept exposure delta ───────────────────────────────────────────
    delta = (steer_exp - base_exp).detach().cpu().numpy()
    base_np  = base_exp.detach().cpu().numpy()
    steer_np = steer_exp.detach().cpu().numpy()

    # rank concepts by absolute exposure change
    order = np.argsort(-np.abs(delta))
    # always include suppressed concepts in the display, even if Δ is small
    display = list(suppressed) + [i for i in order.tolist() if i not in suppressed]
    #display = display[: max(top_changes, len(suppressed))]

    if verbose:
        print(f"\n{'='*72}")
        print(f"Individual steering — suppress concept(s): "
              f"{[(i, name(i)) for i in suppressed]}")
        print(f"{'='*72}")

        print(f"\n  List change (top-{k}):")
        print(f"    RBO={list_metrics['rbo']:.4f}  "
              f"Jaccard={list_metrics['jaccard']:.4f}  "
              f"overlap={list_metrics['overlap']:2d}/{k}  "
              f"changed={list_metrics['changed']:2d}")

        print(f"    {'idx':>4s}  {'concept':<22s}  {'before':>9s}  "
              f"{'after':>9s}  {'delta':>9s}")
        print(f"    {'-'*4}  {'-'*22}  {'-'*9}  {'-'*9}  {'-'*9}")
        for i in display:
            mark = '*' if i in suppressed else ' '
            print(f"   {mark}[{i:3d}] {name(i):<22s}  "
                  f"{base_np[i]:9.4f}  {steer_np[i]:9.4f}  "
                  f"{delta[i]:+9.4f}")
        print(f"{'='*72}\n")

    return {
        'suppressed':   suppressed,
        'baseline_list':  base_list,
        'steered_list':   steer_list,
        'list_metrics':   list_metrics,
        'baseline_exposure': base_np,    # [n_concepts]
        'steered_exposure':  steer_np,   # [n_concepts]
        'delta_exposure':    delta,      # [n_concepts]
    }

In [ ]:
from recbole.quick_start import run_recbole

result = run_recbole(
    model='SASRec',
    dataset='beeradvocate',
    config_file_list=['YAML_files/SASRec_beeradvocate_config.yaml'],
)

In [10]:
from recbole.quick_start import load_data_and_model

config, model, dataset, train_data, valid_data, test_data = load_data_and_model(
    model_file='/home/mvarasteh/post-hoc/saved/SASRec_SAE-ml-1m_New.pth', config_dict={
        #"base_path": "./saved/SASRec-lastfm_final.pth",

        'topk': [10, 20],
        'metrics': ['Recall', 'NDCG', 'MRR', 'ItemCoverage'],
    },
)
## SASRec_SAE-Jul-14-2026_16-35-25.pth. (SAE for SAE)
# model, config, dataset are reconstructed automatically from the checkpoint
# train_data/valid_data/test_data are rebuilt using the same split as training

from recbole.trainer import Trainer  # or your LoggingTrainer
trainer = Trainer(config, model)  # swap in your custom trainer class if needed
trainer.eval_collector.data_collect(train_data)

valid_result = trainer.evaluate(valid_data, load_best_model=False)
test_result = trainer.evaluate(test_data, load_best_model=False)

print('Valid:', valid_result)
print('Test:', test_result)

/home/mvarasteh/post-hoc/recbole/quick_start/quick_start.py:249: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_file, map_location=torch.device(

Valid: OrderedDict([('recall@10', 0.2671), ('recall@20', 0.38), ('ndcg@10', 0.1438), ('ndcg@20', 0.1723), ('mrr@10', 0.1064), ('mrr@20', 0.1142), ('itemcoverage@10', 0.705), ('itemcoverage@20', 0.7989)])
Test: OrderedDict([('recall@10', 0.2616), ('recall@20', 0.3719), ('ndcg@10', 0.1421), ('ndcg@20', 0.17), ('mrr@10', 0.1059), ('mrr@20', 0.1136), ('itemcoverage@10', 0.6971), ('itemcoverage@20', 0.8039)])


In [ ]:
tmp=pd.read_pickle('/home/mvarasteh/post-hoc/dataset/beeradvocate/saved_concept_individual_items.pkl')

In [ ]:
tmp['concept_names']

In [ ]:
x=pd.read_pickle("/home/mvarasteh/post-hoc/dataset/ml-1mm/saved_concept_individual_items.pkl")

In [ ]:
x['concept_names'].index("modern")

In [ ]:
x['concept_names']

In [ ]:
# %% [markdown]
# # Tier concept probe — synthetic pop / mid / niche users through the CBM
#
# Builds synthetic users whose histories are 100% popular / mid / niche items,
# passes them through the trained CBM, and inspects the tier concept
# activations (c_hat) per cohort. Run cells in order.

# %% ── Cell 1: setup & load checkpoint ──────────────────────────────────────
import numpy as np
import torch
import matplotlib.pyplot as plt
from recbole.quick_start import load_data_and_model

CKPT_PATH = "./saved/SASRec_CBM-Jul-20-2026_10-38-21.pth"     # <-- your CBM checkpoint

# Concept layout — ADJUST PER DATASET
TIER_SLICE = (18, 21)                          # tier dims in the concept vector
TIER_NAMES = ["pop", "mid", "niche"]           # order of dims WITHIN the group
                                               # verify with concept_names below!
SEQ_LEN = 50                                   # synthetic history length
N_USERS = 1000                                  # users per cohort
rng = np.random.default_rng(0)

config, model, dataset, train_data, valid_data, test_data = \
    load_data_and_model(model_file=CKPT_PATH)
device = config["device"]
model.eval()
model.steer_concept_idx = None                 # make sure steering is OFF
if hasattr(model, "steer_deltas"):
    model.steer_deltas = None

a, b = TIER_SLICE
print("Tier concept names in cache order:", 
      [model.concept_names[i] for i in range(a, b)])
# ^^^ If this doesn't match TIER_NAMES, fix TIER_NAMES before proceeding.

# %% ── Cell 2: build item pools per tier ────────────────────────────────────
IC = model.item_concepts.cpu().numpy()         # [n_items, n_concepts]
tier_of_item = IC[:, a:b].argmax(1)
tier_of_item[0] = -1                           # index 0 = RecBole padding, exclude

pools = [np.where(tier_of_item == t)[0] for t in range(b - a)]
for name, p in zip(TIER_NAMES, pools):
    print(f"{name:>6}: {len(p)} items")

# %% ── Cell 3: cohort builder + forward pass ────────────────────────────────
def build_cohort(pool, n_users=N_USERS, seq_len=SEQ_LEN):
    """Synthetic users: seq_len items sampled from `pool`, left-aligned,
    zero-padded to max_seq_length (RecBole convention)."""
    max_len = model.max_seq_length
    seqs = torch.zeros((n_users, max_len), dtype=torch.long)
    for u in range(n_users):
        items = rng.choice(pool, size=seq_len, replace=len(pool) < seq_len)
        seqs[u, :seq_len] = torch.from_numpy(items.astype(np.int64))
    lens = torch.full((n_users,), seq_len, dtype=torch.long)
    return seqs.to(device), lens.to(device)

@torch.no_grad()
def run_cohort(pool):
    seqs, lens = build_cohort(pool)
    h, c_hat, z, h_hat = model.forward(seqs, lens)
    return c_hat.cpu().numpy(), z.cpu().numpy()

# %% ── Cell 4: run the three cohorts, capture activations ───────────────────
c_by_cohort = {}       # cohort name -> full c_hat [N_USERS, n_concepts]
z_by_cohort = {}

for t, name in enumerate(TIER_NAMES):
    print(f"t {name}")
    c, z = run_cohort(pools[t])
    c_by_cohort[name] = c
    z_by_cohort[name] = z
    tier_act = c[:, a:b].mean(0)
    winner = TIER_NAMES[int(tier_act.argmax())]
    ok = "OK " if winner == name else f"WRONG (max={winner})"
    print(f"{name:>6} cohort | " +
          " ".join(f"c[{n}]={v:.3f}" for n, v in zip(TIER_NAMES, tier_act)) +
          f"  -> {ok}")

# %% ── Cell 5: 3×3 activation matrix heatmap ────────────────────────────────
M = np.array([c_by_cohort[n][:, a:b].mean(0) for n in TIER_NAMES])

fig, ax = plt.subplots(figsize=(4.2, 3.6))
im = ax.imshow(M, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(3)); ax.set_xticklabels([f"c[{n}]" for n in TIER_NAMES])
ax.set_yticks(range(3)); ax.set_yticklabels([f"{n} users" for n in TIER_NAMES])
ax.set_xlabel("predicted tier concept"); ax.set_ylabel("synthetic cohort")
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center",
                color="white" if M[i, j] < 0.6 else "black")
plt.colorbar(im, label="mean activation")
plt.title("Tier concept selectivity")
plt.tight_layout()
plt.savefig("tier_selectivity_heatmap.pdf", bbox_inches="tight")
plt.show()
# Diagonal-dominant = tier concepts fire for the matching cohort.

# %% ── Cell 6: per-user distributions (not just means) ──────────────────────
# Means can hide bimodality; violin plots show whether individual synthetic
# users are classified crisply.
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharey=True)
for ax, (t, name) in zip(axes, enumerate(TIER_NAMES)):
    data = [c_by_cohort[name][:, a + j] for j in range(3)]
    ax.violinplot(data, showmeans=True)
    ax.set_xticks([1, 2, 3]); ax.set_xticklabels(TIER_NAMES)
    ax.set_title(f"{name} users")
    ax.set_ylim(-0.05, 1.05)
axes[0].set_ylabel("tier concept activation")
plt.tight_layout()
plt.show()

# %% ── Cell 7 (optional): popularity-dose curve ─────────────────────────────
# Mix popular and niche items at varying fractions; tier activations should
# respond monotonically — the property steering relies on.
def build_mixture(frac_pop, n_users=N_USERS, seq_len=SEQ_LEN):
    n_p = int(round(frac_pop * seq_len))
    max_len = model.max_seq_length
    seqs = torch.zeros((n_users, max_len), dtype=torch.long)
    for u in range(n_users):
        p = rng.choice(pools[0], size=n_p, replace=False) if n_p else np.array([], int)
        q = rng.choice(pools[2], size=seq_len - n_p, replace=False) \
            if n_p < seq_len else np.array([], int)
        items = np.concatenate([p, q]); rng.shuffle(items)
        seqs[u, :seq_len] = torch.from_numpy(items.astype(np.int64))
    lens = torch.full((n_users,), seq_len, dtype=torch.long)
    return seqs.to(device), lens.to(device)

fracs = [0.0, 0.25, 0.5, 0.75, 1.0]
curve = []
for f in fracs:
    seqs, lens = build_mixture(f)
    with torch.no_grad():
        _, c_hat, _, _ = model.forward(seqs, lens)
    curve.append(c_hat[:, a:b].mean(0).cpu().numpy())
curve = np.array(curve)

plt.figure(figsize=(5, 3.5))
for j, n in enumerate(TIER_NAMES):
    plt.plot(fracs, curve[:, j], marker="o", label=f"c[{n}]")
plt.xlabel("fraction of popular items in history")
plt.ylabel("mean tier activation")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("tier_dose_curve.pdf", bbox_inches="tight")
plt.show()


print("c[pop] monotone increasing:", bool(np.all(np.diff(curve[:, 0]) > 0)))
print("c[niche] monotone decreasing:", bool(np.all(np.diff(curve[:, 2]) < 0)))

In [ ]:
z_pop   = z_by_cohort["pop"]     # [N, 64]
z_niche = z_by_cohort["niche"]
z_mid   = z_by_cohort["mid"]

mean_diff = z_pop.mean(0) - z_niche.mean(0)

# Effect size per dim (Cohen's d) — the scale-free version
sd_pool = np.sqrt((z_pop.var(0, ddof=1) + z_niche.var(0, ddof=1)) / 2) + 1e-8
d = mean_diff / sd_pool
order = np.argsort(-np.abs(d))
for i in order[:10]:
    print(f"z[{i:2d}]  d={d[i]:+6.2f}   pop={z_pop.mean(0)[i]:+.3f}  niche={z_niche.mean(0)[i]:+.3f}")
print(f"\nmax |d| = {np.abs(d).max():.2f},  dims with |d|>0.5: {(np.abs(d)>0.5).sum()}/64")

# The aggregate that matters most: is popularity a DIRECTION in z-space?
# (uniform norms + no single loud dim ≠ no signal — it can hide as a combination)
u = mean_diff / (np.linalg.norm(mean_diff) + 1e-12)      # candidate popularity direction
proj_pop, proj_niche = z_pop @ u, z_niche @ u
d_dir = (proj_pop.mean() - proj_niche.mean()) / \
        (np.sqrt((proj_pop.var(ddof=1) + proj_niche.var(ddof=1)) / 2) + 1e-8)
print(f"Cohen's d along the difference-of-means direction: {d_dir:.2f}")